# Lumis 07: Skin-tone labels for the test set

Labels every test image into one of three Fitzpatrick groups so test performance can be broken down by skin tone (the fairness analysis). Three checks back up the labels, as decided in the build plan:

1. **Manual labels** on all 977 test images (this notebook, round 1)
2. **Self-consistency:** a blind re-label of 100 images, done 2 to 3 weeks later (this notebook, round 2)
3. **ITA cross-check:** an objective skin-tone angle computed from the pixels and compared with the manual labels (step 5)

The Fitzpatrick17k blind check comes in a separate notebook.

**How to label**
- Judge the **unaffected skin around the lesion**, not the rash itself
- Many DermNet photos are old slides with a **blue or purple color cast**. Mentally correct for it
- Pick **Can't tell** when there is no normal skin visible, or the color is too distorted to judge. Don't guess
- The diagnosis is hidden on purpose, so it can't influence the label
- Every click saves to Drive, so you can stop and resume any time

| Group | Typical appearance |
|---|---|
| **I to II** | Very fair to fair; burns easily |
| **III to IV** | Light brown to moderate brown (most East and Southeast Asian skin) |
| **V to VI** | Dark brown to very dark |

Runtime: CPU is fine.

## 1. Settings

In [ ]:
ROUND = 1      # 1 = all test images; 2 = blind re-label of 100 random test images (do this 2 to 3 weeks after round 1)
SEED = 42

from google.colab import drive
drive.mount('/content/drive')

import os, io, zipfile
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image

BASE = '/content/drive/MyDrive/Lumis'
LOCAL = '/content/dermnet'
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)

split = pd.read_csv(f'{BASE}/data/lumis_split_v2.csv')
test = split[split.split == 'test'].reset_index(drop=True)
if ROUND == 1:
    queue = test.sample(frac=1, random_state=SEED).reset_index(drop=True)      # shuffled, so classes are mixed
else:
    queue = test.sample(100, random_state=SEED + 1).reset_index(drop=True)

LABELS_FILE = f'{BASE}/data/skin_tone_labels_round{ROUND}.csv'
labels = {}
if os.path.exists(LABELS_FILE):
    prev = pd.read_csv(LABELS_FILE)
    labels = dict(zip(prev.path, prev.tone))
print(f'Round {ROUND}: {len(queue)} images to label, {len(labels)} already done')

## 2. Label

Run this cell, then click a button for each image. Use **Undo** if you mis-click.

In [ ]:
import ipywidgets as w
from IPython.display import display

OPTIONS = ['I-II', 'III-IV', 'V-VI', "Can't tell"]
history = []

def save():
    out = pd.DataFrame({'path': list(labels), 'tone': list(labels.values())})
    out = out.merge(test[['path', 'lumis_class']], on='path', how='left')
    out.to_csv(LABELS_FILE, index=False)

def next_index():
    for j, p in enumerate(queue.path):
        if p not in labels:
            return j
    return None

img = w.Image(format='jpeg', width=480)
info = w.HTML()
buttons = []
for opt in OPTIONS:
    b = w.Button(description=opt, layout=w.Layout(width='130px', height='44px'))
    buttons.append(b)
undo = w.Button(description='Undo', button_style='warning', layout=w.Layout(width='90px', height='44px'))

def render():
    j = next_index()
    if j is None:
        img.value = b''
        info.value = f'<h3>All {len(queue)} images labeled. Saved to {LABELS_FILE}</h3>'
        return
    with Image.open(Path(LOCAL) / queue.path[j]) as im:
        im = im.convert('RGB')
        im.thumbnail((480, 480))
        buf = io.BytesIO()
        im.save(buf, 'JPEG', quality=90)
    img.value = buf.getvalue()
    info.value = f'<b>{len(labels)} / {len(queue)} labeled</b>'

def make_handler(opt):
    def handler(_):
        j = next_index()
        if j is None:
            return
        p = queue.path[j]
        labels[p] = opt
        history.append(p)
        save()
        render()
    return handler

for b, opt in zip(buttons, OPTIONS):
    b.on_click(make_handler(opt))

def on_undo(_):
    if history:
        labels.pop(history.pop(), None)
        save()
        render()
undo.on_click(on_undo)

render()
display(w.VBox([info, img, w.HBox(buttons + [undo])]))

## 3. Distribution by class (after labeling)

Shows how many test images fall in each skin-tone group, per condition. Small groups mean wide uncertainty in the fairness results.

In [ ]:
lab = pd.read_csv(LABELS_FILE)
table = pd.crosstab(lab.lumis_class, lab.tone, margins=True, margins_name='total')
display(table)
table.to_csv(f'{BASE}/exports/skin_tone_distribution_round{ROUND}.csv')

## 4. Round 2 only: self-consistency

Compares the blind re-labels with round 1. Run this after finishing round 2.

In [ ]:
from sklearn.metrics import cohen_kappa_score
r1_file, r2_file = f'{BASE}/data/skin_tone_labels_round1.csv', f'{BASE}/data/skin_tone_labels_round2.csv'
if os.path.exists(r2_file):
    m = pd.read_csv(r1_file).merge(pd.read_csv(r2_file), on='path', suffixes=('_r1', '_r2'))
    m = m[(m.tone_r1 != "Can't tell") & (m.tone_r2 != "Can't tell")]
    order = {'I-II': 0, 'III-IV': 1, 'V-VI': 2}
    a, b = m.tone_r1.map(order), m.tone_r2.map(order)
    print(f'{len(m)} images labeled in both rounds (excluding Can\'t tell)')
    print('Exact agreement:', round((a == b).mean(), 3))
    print('Cohen kappa (linear weights):', round(cohen_kappa_score(a, b, weights='linear'), 3))
    display(pd.crosstab(m.tone_r1, m.tone_r2))
else:
    print('Round 2 not done yet: set ROUND = 2 in step 1 in 2 to 3 weeks and label again.')

## 5. ITA cross-check

The Individual Typology Angle, ITA = arctan((L* - 50) / b*), is computed from the median of skin-colored pixels and mapped to the same three groups: I to II above 41 degrees, III to IV from 10 to 41, V to VI below 10.

Expect only moderate agreement: lesions, lighting, and old-slide color casts all distort pixel color. (A gray-world color correction was tested and rejected: on skin-dominated images it pulls skin toward gray and distorts ITA further.) That is why ITA is a cross-check, not the label.

In [ ]:
from skimage.color import rgb2lab
from sklearn.metrics import cohen_kappa_score

def ita_angle(path):
    with Image.open(path) as im:
        rgb = np.asarray(im.convert('RGB').resize((160, 160)), dtype=np.float64)
    ycc = np.asarray(Image.fromarray(rgb.astype(np.uint8)).convert('YCbCr'), dtype=np.int16)
    mask = (ycc[..., 2] >= 130) & (ycc[..., 2] <= 190) & (ycc[..., 1] >= 70) & (ycc[..., 1] <= 130)
    if mask.sum() < 300:
        return np.nan
    lab = rgb2lab(rgb / 255.0)
    L, b = np.median(lab[..., 0][mask]), np.median(lab[..., 2][mask])
    return float(np.degrees(np.arctan2(L - 50, b)))

def ita_group(a):
    if np.isnan(a):
        return None
    return 'I-II' if a > 41 else ('III-IV' if a > 10 else 'V-VI')

lab = pd.read_csv(f'{BASE}/data/skin_tone_labels_round1.csv')
lab['ita'] = [ita_angle(Path(LOCAL) / p) for p in lab.path]
lab['ita_group'] = lab.ita.map(ita_group)
lab.to_csv(f'{BASE}/data/skin_tone_labels_round1_with_ita.csv', index=False)

m = lab[(lab.tone != "Can't tell") & lab.ita_group.notna()]
order = {'I-II': 0, 'III-IV': 1, 'V-VI': 2}
a, b = m.tone.map(order), m.ita_group.map(order)
print(f'{len(m)} images compared')
print('Exact agreement with ITA:', round((a == b).mean(), 3))
print('Cohen kappa (linear weights):', round(cohen_kappa_score(a, b, weights='linear'), 3))
display(pd.crosstab(m.tone, m.ita_group, rownames=['manual'], colnames=['ITA']))

## 6. Remove non-skin images found while labeling

Labeling every test image was also a full manual review of the test set. This shows every image marked **Can't tell**, labeled `c0`, `c1`, and so on, so the ones that are not skin photos at all (microscope slides, an insect with no skin visible) can be removed. Photos of real skin with a tick or louse on it are valid images for the infestations class and stay.

Only the test set changes here; training and validation are untouched, so no retraining is needed.

In [ ]:
import matplotlib.pyplot as plt

lab = pd.read_csv(f'{BASE}/data/skin_tone_labels_round1.csv')
cant = lab[lab.tone == "Can't tell"].reset_index(drop=True)
cant['c'] = range(len(cant))
rows = int(np.ceil(len(cant) / 5))
fig, axes = plt.subplots(rows, 5, figsize=(20, 4 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis('off')
for ax, (_, r) in zip(axes.flat, cant.iterrows()):
    ax.imshow(Image.open(Path(LOCAL) / r.path).convert('RGB'))
    ax.set_title(f'c{r.c}  {Path(r.path).stem[:26]}', fontsize=10)
plt.tight_layout()
plt.show()

Type the `c` numbers of the images that are **not** skin photos. The pre-filled values are the clearly non-skin ones from the 2026-11-11 review; check the rest yourself.

In [ ]:
REMOVE_C = 'c5, c14, c15, c16, c18'

picked = {int(x) for x in REMOVE_C.replace('c', '').replace(' ', '').split(',') if x}
drop = cant[cant.c.isin(picked)]
print('Removing', len(drop), 'test images:')
print(drop[['c', 'lumis_class', 'path']].to_string(index=False))

V2 = f'{BASE}/data/lumis_split_v2.csv'
v2 = pd.read_csv(V2)
v2 = v2[~v2.path.isin(drop.path)]
v2.to_csv(V2, index=False)

REMOVED = f'{BASE}/exports/removed_nonskin_v2.csv'
pd.concat([pd.read_csv(REMOVED),
           drop[['path', 'lumis_class']].assign(split='test', skin_frac=np.nan,
                                                reason='not a skin photo (found during skin-tone labeling)')]
          ).drop_duplicates('path').to_csv(REMOVED, index=False)

for f in [f'{BASE}/data/skin_tone_labels_round1.csv', f'{BASE}/data/skin_tone_labels_round1_with_ita.csv']:
    if os.path.exists(f):
        d = pd.read_csv(f)
        d[~d.path.isin(drop.path)].to_csv(f, index=False)

lab = pd.read_csv(f'{BASE}/data/skin_tone_labels_round1.csv')
table = pd.crosstab(lab.lumis_class, lab.tone, margins=True, margins_name='total')
table.to_csv(f'{BASE}/exports/skin_tone_distribution_round1.csv')
print('Test set now has', (v2.split == 'test').sum(), 'images')
display(table)